In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. Load Silver Customers
# ============================================================

silver_customers_df = spark.table(
    "customer360_dev.silver.customers"
)

target_table = "customer360_dev.gold.dim_customer"


# ============================================================
# 2. Check Existing Gold Rows
# ============================================================

existing_gold_count = (
    spark.table(target_table)
    .count()
)

print(
    "Existing Gold dim_customer rows:",
    existing_gold_count
)

Existing Gold dim_customer rows: 0


In [0]:
# ============================================================
# 3. Build Initial SCD2 Records
# ============================================================

if existing_gold_count == 0:

    initial_scd2_df = (
        silver_customers_df

        .select(
            "customer_id",
            "first_name",
            "last_name",
            "email",
            "phone_number",
            "date_of_birth",
            "gender",
            "signup_date",
            "customer_status",
            "loyalty_tier",
            "preferred_language",
            "updated_at"
        )

        .withColumn(
            "effective_from",
            F.coalesce(
                F.col("updated_at"),
                F.current_timestamp()
            )
        )

        .withColumn(
            "effective_to",
            F.to_timestamp(
                F.lit("9999-12-31 23:59:59")
            )
        )

        .withColumn(
            "is_current",
            F.lit(True)
        )

        .withColumnRenamed(
            "updated_at",
            "source_updated_at"
        )
    )

else:

    initial_scd2_df = None

    print(
        "Baseline load skipped because "
        "dim_customer already contains data."
    )

In [0]:
if initial_scd2_df is not None:

    (
        initial_scd2_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            target_table
        )
    )

    print(
        "Initial Customer SCD2 baseline load completed."
    )

Initial Customer SCD2 baseline load completed.


In [0]:
gold_count = (
    spark.table(target_table)
    .count()
)

current_count = (
    spark.table(target_table)
    .filter(
        F.col("is_current") == True
    )
    .count()
)

silver_count = (
    silver_customers_df.count()
)

print(f"Silver customers       : {silver_count:,}")
print(f"Gold SCD2 rows         : {gold_count:,}")
print(f"Current Gold rows      : {current_count:,}")

Silver customers       : 986,000
Gold SCD2 rows         : 986,000
Current Gold rows      : 986,000
